Git-repo: https://github.com/oggefaderen/CompSci.git
# Contributions
Lovro: ...

Oskar: ...

Uffe: ...

## Part 1: Mixing Patterns and Assortativity

### Setup: Imports and graph construction

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from collections import defaultdict
import ast
import random
from tqdm import tqdm
import numpy as np

In [ ]:
df = pd.read_csv('./D2_temp_papers.csv')
df['author_ids'] = df['author_ids'].apply(ast.literal_eval)

G = nx.Graph()
pair_counts = defaultdict(int)

for author_list in df['author_ids']:
    for i in range(len(author_list)):
        for j in range(i + 1, len(author_list)):
            pair = tuple(sorted([author_list[i].strip(), author_list[j].strip()]))
            pair_counts[pair] += 1

weighted_edgelist = [(a, b, count) for (a, b), count in pair_counts.items()]
G.add_weighted_edges_from(weighted_edgelist)
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

### Load country attributes

In [ ]:
authors_df = pd.read_csv('./final_authors.csv')
country_map = dict(zip(authors_df['id'], authors_df['country_code']))

for node in G.nodes():
    G.nodes[node]['country'] = country_map.get(node, None)

countries_assigned = sum(1 for n in G.nodes() if G.nodes[n]['country'] is not None)
print(f"Country assigned to {countries_assigned}/{G.number_of_nodes()} nodes")

## Q1: Country Assortativity Coefficient

Calculate the Assortativity Coefficient for the network based on the country of each node. Implement the calculation using the formula provided during the lecture (Newman equation 2). Do not use the NetworkX implementation.

In [ ]:
# Q1: Country assortativity using Newman equation 2 (mixing matrix approach)
# e_ij = fraction of edges connecting country i to country j
# r = (Tr(e) - ||e^2||) / (1 - ||e^2||)

edge_type_counts = defaultdict(int)
total_edges = 0

for u, v in G.edges():
    c_u = G.nodes[u].get('country')
    c_v = G.nodes[v].get('country')
    if c_u is None or c_v is None:
        continue
    # Treat as undirected: always store both orderings
    edge_type_counts[(c_u, c_v)] += 1
    if c_u != c_v:
        edge_type_counts[(c_v, c_u)] += 1
    total_edges += 1

countries = sorted(set(c for (c, _) in edge_type_counts))
M = 2 * total_edges  # denominator: counts each edge in both directions

# e_ij fraction
e = {(c_i, c_j): edge_type_counts.get((c_i, c_j), 0) / M for c_i in countries for c_j in countries}

# Trace and marginal sums
trace_e = sum(e[(c, c)] for c in countries)
a = {c: sum(e[(c, c2)] for c2 in countries) for c in countries}
sum_a_sq = sum(a[c]**2 for c in countries)

original_country_r = (trace_e - sum_a_sq) / (1 - sum_a_sq)
print(f"Country assortativity r = {original_country_r:.4f}")